# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Мои предсказания и объяснения:

1. 1 и 1.3333333333333333 потому что пропуск один, а fillna(0) ставит вместо него 0 и среднее (1+0+3)/3
2. 2 потому что strip убирает пробелы а lower делает A маленькой, остаются только a и b
3. 3 потому что drop_duplicates возвращает новую таблицу а мы ее не сохранили в df, поэтому в df так и 3 строки
4. 1005 потому что из "1 000" убираем пробел, to_numeric делает числа и складывается 1000 + 5
5. [9, 10] потому что to_datetime превращает текст в даты а dt.month берет номер месяца, сентябрь 9 и октябрь 10

In [19]:
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

1 1.3333333333333333
2
3
1005
[9, 10]


In [20]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("../hw06/orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("../hw06/orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [21]:
# База 1
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print(f"Исходно строк: {len(df)}")

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print(f"После перевода сумм: {len(df)}")

    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print(f"После очистки городов: {len(df)}")

    df = df.drop_duplicates()
    print(f"После удаления дублей: {len(df)}")

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print(f"После перевода дат: {len(df)}")

    df = df[df["customer_id"] != "TEST"]
    print(f"После удаления TEST: {len(df)}")
    print(f"Пропусков в amount: {df['amount'].isna().sum()}, в city: {df['city'].isna().sum()}, order_id уникален: {df['order_id'].is_unique}")
    return df

df = clean("../hw06/orders.csv")
print(f"Чистых заказов: {len(df)}, из них отмененных {(df['status'] == 'отменён').sum()}")

Исходно строк: 5463
После перевода сумм: 5463
После очистки городов: 5463
После удаления дублей: 5283
После перевода дат: 5283
После удаления TEST: 5280
Пропусков в amount: 0, в city: 0, order_id уникален: True
Чистых заказов: 5280, из них отмененных 541


В чистой таблице 5280 заказов. Убрали 180 дублей и 3 TEST заказа, отмененные оставили чтобы потом считать долю отмен

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [22]:
# База 2
paid = df["status"] == "оплачен"
sep = df.loc[paid & (df["month"] == 9), "amount"].sum()
octo = df.loc[paid & (df["month"] == 10), "amount"].sum()
print(f"Вся выручка: сентябрь {sep}, октябрь {octo}, изменение {(octo - sep) / sep:+.2%}")

for c in df["city"].unique():
    s = df.loc[paid & (df["city"] == c) & (df["month"] == 9), "amount"].sum()
    o = df.loc[paid & (df["city"] == c) & (df["month"] == 10), "amount"].sum()
    print(f"{c}: сентябрь {s}, октябрь {o}, изменение {(o - s) / s:+.2%}")

Вся выручка: сентябрь 35243410.0, октябрь 34419520.0, изменение -2.34%
Алматы: сентябрь 14501070.0, октябрь 14643020.0, изменение +0.98%
Астана: сентябрь 9034220.0, октябрь 8850530.0, изменение -2.03%
Караганда: сентябрь 3648520.0, октябрь 3371680.0, изменение -7.59%
Шымкент: сентябрь 4558650.0, октябрь 4044010.0, изменение -11.29%
Атырау: сентябрь 923990.0, октябрь 969860.0, изменение +4.96%
Актобе: сентябрь 2576960.0, октябрь 2540420.0, изменение -1.42%


Выручка по городам совпадает с таблицей город × месяц из урока 4 (Шымкент 4 558 650 → 4 044 010, Алматы 14 501 070 → 14 643 020 и тд). На чистых данных вся выручка упала всего на 2.34% а не на 22.7%. Сильнее всех падает Шымкент -11.29%, потом Караганда -7.59%, Алматы и Атырау даже выросли

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [23]:
# База 3
def change(d):
    p = d["status"] == "оплачен"
    s = d.loc[p & (d["order_date"] < "2026-10"), "amount"].sum()
    o = d.loc[p & (d["order_date"] >= "2026-10"), "amount"].sum()
    return (o - s) / s

raw = pd.read_csv("../hw06/orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))
nodup = raw.drop_duplicates()

print(f"Как считал менеджер: {len(raw)} строк, {change(raw):+.1%}")
print(f"Без дублей: {len(nodup)} строк (убрали {len(raw) - len(nodup)}), {change(nodup):+.1%}")
print(f"Без TEST: {len(df)} строк (убрали {len(nodup) - len(df)}), {change(df):+.1%}")

Как считал менеджер: 5463 строк, -22.7%
Без дублей: 5283 строк (убрали 180), -19.6%
Без TEST: 5280 строк (убрали 3), -2.3%


| Шаг очистки | Убрали строк | Падение выручки октябрь к сентябрю |
| --- | --- | --- |
| Как считал менеджер | 0 | -22.7% |
| Суммы в числа, города, даты | 0 | -22.7% |
| Убрали дубли | 180 | -19.6% |
| Убрали TEST | 3 | -2.3% |

Сильнее всего цифру исказили 3 TEST заказа по 9 999 990: из за них падение было -22.7% а на самом деле -2.3%

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [24]:
# Стандарт 4
no_days = df["delivery_days"].isna()
print(f"Пропусков в delivery_days: {no_days.sum()}")
print(df.loc[no_days, "city"].value_counts())
print(df.loc[no_days, "status"].value_counts())

sh_days = df.loc[df["city"] == "Шымкент", "delivery_days"]
print(f"Доставка за 2 дня в Шымкенте (dropna): {(sh_days.dropna() <= 2).mean():.1%}")
print(f"Доставка за 2 дня в Шымкенте (fillna медиана): {(sh_days.fillna(sh_days.median()) <= 2).mean():.1%}")
print(f"Доставка за 2 дня в Шымкенте (fillna 0): {(sh_days.fillna(0) <= 2).mean():.1%}")

Пропусков в delivery_days: 158
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
status
оплачен    144
отменён     14
Name: count, dtype: int64
Доставка за 2 дня в Шымкенте (dropna): 29.8%
Доставка за 2 дня в Шымкенте (fillna медиана): 29.0%
Доставка за 2 дня в Шымкенте (fillna 0): 31.8%


Пропусков 158, больше всего в Алматы и Астане потому что там больше всего заказов, по статусу почти все у оплаченных. Честный вариант dropna 29.8% потому что считаем только заказы где срок известен. fillna(медиана) придумывает срок 5 дней и доля падает до 29.0%, а fillna(0) хуже всего, 0 дней это быстрее любой доставки и доля растет до 31.8%

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [25]:
# Стандарт 5

df2 = pd.read_csv("../hw06/orders.csv")
df2 = df2.drop_duplicates()
df2["city"] = df2["city"].str.strip().str.lower().map(CITY)
print(f"Городов: {df2['city'].nunique()}, строк: {len(df2)}")

Городов: 6, строк: 5283


Ошибка 1: drop_duplicates возвращает новую таблицу, без df = дубли не удаляются и строк так и 5463
# Ошибка 2: lower без strip и без словаря, " Алматы" и Almaty остаются отдельными городами

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [26]:
# Стандарт 6
days = df["order_date"].dt.day_name().value_counts()
print(days)
print(f"Меньше всего заказов в {days.idxmin()}: {days.min()}, больше всего в {days.idxmax()}: {days.max()}")
print(f"В {days.idxmin()} заказов меньше на {(days.max() - days.min()) / days.max():.2%}")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Меньше всего заказов в Monday: 684, больше всего в Thursday: 786
В Monday заказов меньше на 12.98%


Меньше всего заказов в понедельник (684) и воскресенье (697), больше всего в четверг (786). В понедельник заказов на 12.98% меньше чем в четверг, значит в понедельник и воскресенье можно ставить меньше курьеров а со вторника по субботу больше

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [30]:
# Челлендж 7
sh = df[df["city"] == "Шымкент"]
print(f"Средний срок в Шымкенте: сентябрь {sh.loc[sh['month'] == 9, 'delivery_days'].mean():.2f}, октябрь {sh.loc[sh['month'] == 10, 'delivery_days'].mean():.2f}")

fast = sh[sh["delivery_days"] < 5].sort_values("order_date")
last_fast = fast["order_date"].iloc[-1]
slow = sh[(sh["delivery_days"] >= 5) & (sh["order_date"] > last_fast)].sort_values("order_date")
print(f"Последний быстрый заказ: {last_fast}, первый медленный: {slow['order_date'].iloc[0]}")

other = df[df["city"] != "Шымкент"]
print(f"Другие города: сентябрь {other.loc[other['month'] == 9, 'delivery_days'].mean():.2f}, октябрь {other.loc[other['month'] == 10, 'delivery_days'].mean():.2f}")

Средний срок в Шымкенте: сентябрь 2.03, октябрь 6.54
Последний быстрый заказ: 2026-09-30 22:00:00, первый медленный: 2026-10-01 10:00:00
Другие города: сентябрь 2.00, октябрь 2.01


В сентябре Шымкент доставлял за 2.03 дня как все города, а с 1 октября (последний быстрый заказ 30.09 в 22:00, первый медленный 01.10 в 10:00) доставка идет 5-8 дней, в среднем 6.54. В других городах ничего не поменялось, 2.00 и 2.01 дня. Значит сломалась доставка именно в Шымкенте 1 октября, в первую очередь надо проверить что там поменялось: курьеры, склад или подрядчик

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [29]:
# Челлендж 8
# errors="coerce" превращает в NaN то что не получилось перевести в число ("нет данных")
def to_amount(s):
    return pd.to_numeric(s.str.replace(" ", "").str.replace(",", "."), errors="coerce")

print(to_amount(pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])))
total = to_amount(pd.read_csv("../hw06/orders.csv")["amount"]).sum()
print(f"Сумма amount: {total}, совпадает с уроком: {total == 109883360}")

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64
Сумма amount: 109883360.0, совпадает с уроком: True
